# DAY-5: Finalizing The AI customer assitant for an Airline.

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI
import sqlite3
import gradio as gr
import json

In [2]:
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("Key loaded")
else:
    print("Key not loaded")

openai = OpenAI()
Model = "gpt-4.1-mini"
DB = "prices.db"

Key loaded


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def get_ticket_price(city):
    print(f"DataBase query: requested price for the ticket to the following city {city}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price From prices WHERE city = ?",(city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to the {city} is : {result[0]}" if result else "No price data available!"

In [5]:
get_ticket_price("chennai")

DataBase query: requested price for the ticket to the following city chennai


'Ticket price to the chennai is : 14000.0'

In [6]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [7]:
def chat(message, history):
    messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(
        model = Model,
        messages = messages,
        tools =tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(modle = Model, messages = messages, tools= tools)
    return response.choices[0].message.content

In [8]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses


In [9]:
# gr.ChatInterface(fn = chat, type = "messages").launch()

# Multi-Modal Time!!

* Generating an image using GPT model with the help of OpenAI library.

In [10]:
import base64
from io import BytesIO
from PIL import Image


In [11]:
def artist(city):
    image_response = openai.images.generate(
        model = "gpt-image-1-mini",
        prompt = f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant style.",
        size = "1024x1024",
        n=1,
    )

    image_b64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_b64)
    return Image.open(BytesIO(image_data))

In [12]:
# image = artist("Delhi")
# display(image)

## Function to create AI generated TTS:

In [13]:
def talk(message):
    response = openai.audio.speech.create(
        model = "gpt-4o-mini-tts",
        voice = "onyx",
        input = message
    )

    return response.content

## Project : Creating a Fully working Airline Booking assistant with AI generated Voice and Images:

In [ ]:
def chat(history):
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=Model, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=Model, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talk(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [15]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

### In Gradio, there are 3 main Interface:
> we have used gr.Interface, gr.ChatInterface. But for this we are using gr.Block, which allows as to create custom Interface layout.

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("Root", "Root@1234"))

C:\Users\yaduk\AppData\Local\Temp\ipykernel_22936\968348099.py:10: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500, type="messages")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


DataBase query: requested price for the ticket to the following city Mumbai
